# 제4장. 원인과 결과 찾기 (원인-결과 찾기 기초)

## 학습 목표
1. "함께 움직인다"와 "원인이 된다"의 차이를 구분하고, 가짜 관계를 찾아낼 수 있다
2. 원인-결과 지도(DAG)를 그리고, 숨은 원인/중간 다리/합류점을 구분할 수 있다
3. "직접 조작"과 "만약 ~했다면?" 사고법을 이해하고 기획에 활용할 수 있다
4. 실험처럼 비교하는 방법(DID, RDD)의 기본 원리를 이해할 수 있다
5. Python으로 숨은 원인을 고려하여 진짜 효과를 계산할 수 있다

---

### 강의 구조 (3시간)

| 시간 | 파트 | 내용 |
|------|------|------|
| | Part 1 | 함께 움직이는 것 vs 원인이 되는 것 (가짜 관계, 역설적 상황) + 조사 과제 |
| | Part 2 | 원인-결과 지도 그리기 (숨은 원인, 중간 다리, 합류점) + 조사 과제 |
| | 휴식 | |
| | Part 3 | 진짜 효과 계산하기 (직접 조작, 만약 ~했다면?, 실험처럼 비교) + 조사 과제 |
| | Part 4 | 실습: 마케팅 캠페인 효과 시뮬레이션 |
| | Part 5 | 종합 실습: 원인-결과 지도 + 효과 계산 |

---

In [1]:
# ============================================================
# 환경설정 및 라이브러리 로드
# ============================================================
# 처음 사용 시 아래 명령을 터미널에서 먼저 실행하세요:
#   python setup_env.py
# 그 후 VSCode 우측 상단에서 커널을 'AI 기획 강의 (Python 3)'으로 선택하세요.
# ============================================================

import sys

# 패키지 확인
_required = ["numpy", "pandas", "plotly", "scipy"]
_missing = []
for _pkg in _required:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"❌ 누락된 패키지: {', '.join(_missing)}")
    print("   터미널에서 다음 명령을 실행하세요: python setup_env.py")
    raise SystemExit(1)

# 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ 라이브러리 로드 완료!")


✅ 라이브러리 로드 완료!


---

## Part 1. 함께 움직이는 것 vs 원인이 되는 것

---

## 5.1 왜 구분이 중요한가?

### 핵심 차이

| 구분 | 함께 움직임 (상관관계) | 원인이 됨 (인과관계) |
|------|----------------------|---------------------|
| 뜻 | 같이 오르락내리락 함 | 한쪽이 다른 쪽을 **만들어냄** |
| 방향 | 양쪽 다 가능 (X ↔ Y) | 한쪽 방향만 (X → Y) |
| 조작했을 때 | 변화 없을 수도 있음 | **변화 생김** (X를 바꾸면 Y가 변함) |
| 기획 활용 | 지켜보기만 (제한적) | 정책 만들기 (**핵심**) |
| 확인 방법 | 데이터로 패턴 찾기 | 실험하거나 원인 찾기 |

### 기획에서 왜 중요할까?

> 기획의 핵심 질문: **"이걸 하면 저게 될까?"**  
> 이 질문에 제대로 답하려면 단순히 "함께 움직이는 것"이 아니라 **"진짜 원인"**을 찾아야 한다.

### 착각하면 생기는 문제

1. 효과 없는 일에 돈과 시간을 **버린다**
2. 진짜 문제를 못 찾아서 **계속 똑같은 문제**가 생긴다
3. 예상 못한 **나쁜 결과**가 생긴다

**예시:** "아이스크림이 많이 팔리면 익사 사고가 늘어난다"는 데이터를 보고 아이스크림 판매를 막으면? → 효과 없음. 진짜 원인은 더운 날씨!

---


In [3]:
# 가짜 관계 시뮬레이션: 아이스크림 판매와 익사 사고
np.random.seed(42)

months = np.arange(1, 13)
# 교란변수: 기온 (여름에 높음)
temperature = 5 + 20 * np.sin((months - 4) * np.pi / 6) + np.random.normal(0, 2, 12)
temperature = np.clip(temperature, -5, 38)

# 아이스크림 판매: 기온에 영향 받음
ice_cream = 50 + 3 * temperature + np.random.normal(0, 5, 12)

# 익사 사고: 기온에 영향 받음 (수영 증가)
drowning = 2 + 0.3 * temperature + np.random.normal(0, 1, 12)
drowning = np.clip(drowning, 0, None)

# 상관계수 계산
corr_ice_drown = np.corrcoef(ice_cream, drowning)[0, 1]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'Spurious Correlation (r = {corr_ice_drown:.2f})',
        'True Causal Structure'
    )
)

# 왼쪽: 상관관계 산점도
fig.add_trace(
    go.Scatter(
        x=ice_cream, y=drowning, mode='markers+text',
        marker=dict(size=12, color='white', line=dict(color='black', width=1)),
        text=[f'M{m}' for m in months],
        textposition='top center',
        textfont=dict(size=12, color='black'),
        name='Monthly Data'
    ),
    row=1, col=1
)
fig.update_xaxes(title_text='Ice Cream Sales (units)', row=1, col=1)
fig.update_yaxes(title_text='Drowning Incidents', row=1, col=1)

# 오른쪽: DAG 구조
# 노드
dag_nodes = [
    {'x': 2, 'y': 3, 'text': 'Temperature\n(Confounder)', 'color': 'black'},
    {'x': 1, 'y': 1, 'text': 'Ice Cream\nSales', 'color': 'black'},
    {'x': 3, 'y': 1, 'text': 'Drowning\nIncidents', 'color': 'black'}
]

for node in dag_nodes:
    fig.add_trace(
        go.Scatter(
            x=[node['x']], y=[node['y']], mode='markers+text',
            marker=dict(size=50, color='white', line=dict(color='black', width=2)),
            text=[node['text']], textposition='middle center',
            textfont=dict(size=9, color='black'),
            showlegend=False
        ),
        row=1, col=2
    )

# 화살표
for ax, ay, x, y in [(2, 2.7, 1.2, 1.3), (2, 2.7, 2.8, 1.3)]:
    fig.add_annotation(
        x=x, y=y, ax=ax, ay=ay,
        xref='x2', yref='y2', axref='x2', ayref='y2',
        showarrow=True, arrowhead=2, arrowsize=1.5, arrowcolor='gray'
    )

# X 표시 (직접 인과관계 없음)
fig.add_annotation(
    x=2, y=0.8, text='❌ No Direct Cause',
    xref='x2', yref='y2', showarrow=False,
    font=dict(size=11, color='red')
)

fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=2)
fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=2)

fig.update_layout(height=400, showlegend=False,
                  title_text='Spurious Correlation: Ice Cream Sales vs Drowning')
fig.show()

print(f"💡 아이스크림 판매와 익사 사고의 상관계수: r = {corr_ice_drown:.2f} (강한 정적 상관)")
print("   그러나 아이스크림이 익사를 유발하는 것이 아니다!")
print("   기온(교란변수)이 두 변수 모두에 영향을 미치는 가짜 관계")


💡 아이스크림 판매와 익사 사고의 상관계수: r = 0.97 (강한 정적 상관)
   그러나 아이스크림이 익사를 유발하는 것이 아니다!
   기온(교란변수)이 두 변수 모두에 영향을 미치는 가짜 관계


### 가짜 관계 (Spurious Correlation)

두 변수 사이에 직접적인 인과관계가 없음에도 상관관계가 관찰되는 현상.  
대부분 **교란변수(Confounder)**에 의해 발생한다.

```
        기온 (교란변수)
       ↙        ↘
아이스크림 판매    익사 사고
```

### 비즈니스 맥락의 가짜 관계

| 관찰된 상관 | 성급한 결론 | 실제 교란변수 |
|------------|------------|------------|
| 마케팅비↑ → 매출↑ | 마케팅 투자가 매출을 높인다 | **성수기** (수요↑ → 마케팅↑ + 매출↑) |
| 교육수준↑ → 성과↑ | 학력이 높으면 성과가 좋다 | **문제해결 능력** |
| 디지털역량↑ → 수익↑ | 디지털 투자하면 수익 증가 | **혁신적 기업문화** 또는 **역인과** |

---


### Simpson's Paradox (심슨의 역설)

전체로 보면 A가 B보다 좋은데, 세부적으로 나눠보면 **모든 그룹에서 B가 더 좋은** 놀라운 현상!

**쉬운 비유:**  
- 전체 평균 성적: 1반(74점)이 2반(71점)보다 높음  
- 하지만 남학생끼리 비교: 1반(75점) < 2반(80점) ✓  
- 여학생끼리 비교: 1반(65점) < 2반(70점) ✓  
- **왜?** 1반에는 남학생이 90명, 여학생이 10명 / 2반에는 남학생이 10명, 여학생이 90명
- 즉, **1반은 (점수가 더 높은) 남학생 비율이 압도적**이고, **2반은 (점수가 더 낮은) 여학생 비율이 압도적**이기 때문!

**숫자로 확인:**
- 1반 전체 평균 = 75×90 + 65×10 = 6750 + 650 = 7400 ÷ 100 = **74점**
- 2반 전체 평균 = 80×10 + 70×90 = 800 + 6300 = 7100 ÷ 100 = **71점**

→ 그룹별(남/여) 비교에서는 2반이 더 좋지만, 그룹 구성 비율 차이 때문에 전체에서는 1반이 더 높게 나옴!

**교훈:**
1. 전체 평균만 보면 **착각**할 수 있다
2. 어떻게 나눠볼지는 **원인과 결과의 관계**를 이해해야 안다
3. **그룹 구성 비율**이 다르면 전체와 부분의 결론이 정반대가 될 수 있다

### 교란변수 식별 체크리스트

| 질문 | 설명 |
|------|------|
| 시간적 선행성 | 처치 이전에 존재하며, 처치와 결과 모두에 영향을 미치는 변수는? |

| 선택 메커니즘 | 처치를 받는 사람/조직은 어떻게 결정되는가? |---

| 대안적 설명 | 관찰된 관계를 설명할 수 있는 제3의 요인은? |

| 문헌 검토 | 선행 연구에서 알려진 교란변수는? || 전문가 의견 | 전문가들이 중요하다고 생각하는 배경 변수는? |


---

## Part 2. 원인-결과 지도 그리기 (DAG)

---

## 5.2 인과 다이어그램 (DAG)

### 기본 개념

**DAG** = 원인과 결과의 흐름을 화살표로 그린 지도

- **D**irected (방향이 있음): 화살표 → 로 원인 → 결과 방향을 표시
- **A**cyclic (다시 돌아오지 않음): A가 B의 원인이면, B는 A의 원인이 될 수 없음 (순환 금지)
- **G**raph: 점과 화살표로 연결된 그림

- 원인과 결과를 쉽게 보기 위한 그림
- **점(Node)**: 각각의 변수 (예: 날씨, 매출 등)
- **화살표(Edge)**: 원인 → 결과를 표시 (X → Y = "X가 Y를 만듦")
- **방향이 있음**: 누가 원인이고 누가 결과인지 명확함
- **돌고 돌지 않음**: A→B→C 가능, 하지만 C→A는 불가 (시간 순서대로)

### 3가지 핵심 패턴

| 패턴 | DAG | 통제 필요 | 통제 시 결과 |
|------|-----|----------|------------|
| **숨은 원인 (교란변수)** | X ← Z → Y | ✅ 필수 | 가짜 관계 제거 |
| **중간 다리 (매개변수)** | X → M → Y | 상황에 따라 | 중간 단계 효과 제거 |
| **합류점 (충돌변수)** | X → Z ← Y | ❌ 금지 | Collider Bias 발생 |

---


In [2]:
# DAG 패턴 시각화 함수
def draw_dag(nodes, edges, title, annotations=None):
    """
    간단한 DAG를 Plotly로 시각화한다.
    
    Args:
        nodes: list of dict {'x', 'y', 'label', 'color'}
        edges: list of tuples (from_idx, to_idx)
        title: 그래프 제목
        annotations: 추가 주석 list
    """
    fig = go.Figure()
    
    # 노드 그리기
    for node in nodes:
        fig.add_trace(go.Scatter(
            x=[node['x']], y=[node['y']],
            mode='markers+text',
            marker=dict(size=55, color='white', line=dict(color='black', width=2)),
            text=[node['label']],
            textposition='middle center',
            textfont=dict(size=10, color='black'),
            showlegend=False
        ))
    
    # 엣지 (화살표)
    for frm, to in edges:
        fig.add_annotation(
            x=nodes[to]['x'], y=nodes[to]['y'],
            ax=nodes[frm]['x'], ay=nodes[frm]['y'],
            xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=2, arrowsize=1.5,
            arrowcolor='gray', arrowwidth=2
        )
    
    if annotations:
        for ann in annotations:
            fig.add_annotation(**ann)
    
    fig.update_layout(
        title=title, height=350, width=500,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False,
                  scaleanchor='x')
    )
    return fig

print("✅ DAG 시각화 함수 정의 완료!")


✅ DAG 시각화 함수 정의 완료!


### 패턴 1: 숨은 원인 (교란변수, Confounder) — 반드시 통제 ✅

```
          성수기 (Z)
         ↙       ↘
     광고비 (X) → 매출 (Y)
```

**한 줄 요약:** 뒤에서 X와 Y **둘 다**를 움직이는 숨은 변수(Z)가 있다.

**예시: "광고비를 많이 쓴 달에 매출이 높다"**

어떤 기업의 월별 데이터를 보니 광고비와 매출이 함께 올라갑니다. "광고를 늘리면 매출이 오르겠구나!" 정말 그럴까요?

| 시기 | 성수기가 하는 일 | 결과 |
|------|-----------------|------|
| 성수기 | 기업이 광고비를 늘림 (Z→X) | 광고비↑ |
| 성수기 | 소비자 수요가 자연히 올라감 (Z→Y) | 매출↑ |
| 비수기 | 기업이 광고비를 줄임 | 광고비↓ |
| 비수기 | 소비자 수요가 자연히 내려감 | 매출↓ |

광고비와 매출이 함께 움직이는 건 **성수기라는 숨은 원인** 때문입니다. 이 데이터만 보고 "비수기에도 광고비를 크게 늘리자!"라고 결정하면? 효과가 기대보다 훨씬 작을 수 있습니다.

**해결법:** 성수기/비수기를 **같은 조건으로 나눈 뒤** 각각에서 광고 효과를 비교합니다. 성수기끼리, 비수기끼리 비교하면 성수기 효과가 제거되어 광고의 **진짜 효과**만 남습니다.

> **핵심 원칙:** 교란변수는 **반드시 통제**(같은 조건으로 만들기)해야 합니다.  
> 통제하지 않으면 진짜 효과를 과대 또는 과소추정합니다.

---

### 패턴 2: 중간 다리 (매개변수, Mediator) — 목적에 따라 다름

```
직원교육 (X) ──→ 직원역량 (M) ──→ 고객만족도 (Y)
     │                                  ↑
     └────────── 직접 효과 ──────────────┘
```

**한 줄 요약:** X가 Y에 영향을 미치는 **중간 경로**에 있는 변수(M)이다.

**예시: "직원교육을 했더니 고객만족도가 올랐다"**

회사가 직원교육 프로그램을 실시했더니 고객만족도가 올라갔습니다. 어떻게 올라간 걸까요? 두 가지 경로가 있습니다.

- **간접 효과 (X→M→Y):** 교육 → 직원 전문 역량 향상 → 더 좋은 고객 응대 → 만족도↑
- **직접 효과 (X→Y):** 교육을 받은 것 자체가 직원의 자신감과 서비스 태도를 바꿈 → 만족도↑

**언제 통제하고, 언제 안 하나요?**

| 알고 싶은 것 | 직원역량(M) 통제? | 이유 |
|---|---|---|
| 교육의 **전체** 효과 | ❌ 안 함 | 역량 향상 경로까지 포함해야 하니까 |
| 교육의 **직접** 효과만 | ✅ 함 | 역량이 같은 직원끼리 비교 → 간접 경로 차단 |

**쉬운 비유:** 서울→부산 가는 길이 두 갈래입니다.
- 고속도로 (간접 효과: 교육→역량→만족)
- 국도 (직접 효과: 교육→만족)
- **전체 교통량**을 알려면 두 길 다 열어둡니다
- **국도만의 교통량**을 알려면 고속도로를 막고(M 통제) 봅니다

> **주의:** 매개변수를 **잘못 통제하면** 전체 효과를 작게 봅니다.  
> 교육이 역량을 높여서 만족도를 올린 부분(간접 효과)이 사라지기 때문입니다.

---

### 패턴 3: 합류점 (충돌변수, Collider) — 인과효과 추정 시 통제 금지 ❌

```
제품 품질 (X) ──→ 시장 생존 (Z) ←── 마케팅 역량 (Y)
```

**한 줄 요약:** 두 변수(X, Y)가 **하나의 결과(Z)에 동시에 영향**을 준다. 교란변수와 화살표 방향이 **정반대**이다.

| | 교란변수 | 충돌변수 |
|---|---|---|
| 구조 | Z → X, Z → Y | X → Z, Y → Z |
| Z의 역할 | X, Y의 **원인** | X, Y의 **결과** |
| 통제 | ✅ 해야 함 | ❌ 하면 안 됨 |

**예시: "살아남은 기업에서 품질과 마케팅이 반비례한다?"**

1,000개 기업의 데이터를 봅시다. 제품 품질과 마케팅 역량은 **원래 아무 관계가 없습니다.**

그런데 **"시장에서 살아남은 기업만"** 골라서 분석하면 어떻게 될까요?

| 기업 유형 | 품질 | 마케팅 | 생존? |
|----------|------|--------|------|
| A사 | 높음 | 낮음 | ✅ 품질로 생존 |
| B사 | 낮음 | 높음 | ✅ 마케팅으로 생존 |
| C사 | 높음 | 높음 | ✅ 둘 다 좋아서 생존 |
| D사 | 낮음 | 낮음 | ❌ 퇴출 (분석에서 빠짐!) |

D사처럼 둘 다 부족한 기업은 퇴출되어 분석에서 빠집니다. 남은 기업들만 보면:
- 품질이 높은 기업 → 마케팅이 약해도 살아남은 것 → **마케팅↓처럼 보임**
- 마케팅이 강한 기업 → 품질이 약해도 살아남은 것 → **품질↓처럼 보임**

결과: **"품질이 좋으면 마케팅이 약하다"는 가짜 관계가 나타남!**

이것이 바로 **생존자 편향(Survivorship Bias)** — 충돌변수 편향의 대표적 형태입니다.

> **핵심 원칙:** 인과효과 추정 시 충돌변수를 **절대 통제하면 안 됩니다.**  
> 통제하면 **없던 가짜 관계**가 만들어집니다.  
> 살아남은 기업만 보고 "마케팅에 투자하면 품질이 떨어진다"고 결론 내리면 완전히 잘못된 판단입니다.

아래 시뮬레이션으로 이 현상을 숫자로 확인해봅시다.

---

실무에서 충돌변수 편향은 다양한 형태로 나타납니다.

대표적인 사례는 다음과 같습니다.

1. **생존 기업만 분석**하면 실패 요인을 알 수 없습니다.  
   이것이 **생존자 편향**입니다. 2차 세계대전 당시 귀환한 폭격기의 피탄 위치만 보고 장갑을 보강하려 했지만, 실제로는 **귀환하지 못한 비행기들이 맞은 부위**가 더 치명적이었습니다.

2. **응답자만 분석**하면 비응답자의 특성을 놓칩니다.  
   이것이 **응답 편향**입니다. 만족도 조사에 응답하는 사람은 매우 만족하거나 매우 불만족한 **극단적 집단**일 수 있습니다.

3. **특정 채널 고객만 분석**하면 전체 고객 특성을 오해할 수 있습니다.  
   예를 들어 앱 사용자만 분석하면 **디지털 친화적 고객 편향**이 발생합니다.

---

## Part 3. 진짜 효과 계산하기

---

## 5.3 지켜보기 vs 직접 조작하기

### do-연산자 (Judea Pearl)

| 구분 | 지켜보기 (관찰) | 직접 조작하기 (개입) |
|------|------|------|
| 표기 | P(Y \| X=x) | P(Y \| **do**(X=x)) |
| 의미 | X가 x인 것을 **발견**했을 때 Y | X를 x로 **강제로 만들었을 때** Y |
| 예시 | 약 먹은 사람들의 회복률 | 약을 **먹게 했을 때** 회복률 |
| DAG 조작 | 변화 없음 | X로 들어오는 화살표 **제거** |

```
원래 DAG:           개입 후 DAG (do(X=x)):
Z → X → Y           Z   X=x → Y
Z → Y               Z → Y

do(X=x)는 Z → X 화살표를 제거
→ 교란변수 Z의 영향을 차단
```

### 백도어 조정 (숨은 뒷길 막기)

관찰 데이터만 있을 때 '직접 조작' 효과를 계산하는 방법:

**P(Y|do(X=x)) = Σ_z P(Y|X=x, Z=z) × P(Z=z)**

→ 숨은 원인(Z)을 같은 조건으로 나눠서 가중 평균을 내면, 관찰한 데이터만으로도 진짜 효과를 알 수 있어요!

---


### 지켜보기 vs 직접 조작하기: 왜 다른가?

```
[지켜보기 — 관찰]                [직접 조작 — 개입]

   Z ──→ X ──→ Y                Z     X=x ──→ Y
   │           ↑                 │            ↑
   └───────────┘                 └────────────┘
  (Z가 X와 Y 둘 다 영향)        (Z→X 끊김! Z는 Y에만 영향)
```

**쉬운 비유: 우산과 비**

> **관찰:** "우산을 들고 다니는 사람은 비를 자주 맞는다" → 우산이 비를 부르나?  
> **개입:** "동전 던지기로 우산을 줄 사람/안 줄 사람을 **무작위 배정**하면 비가 더 오는가?" → 당연히 아니다!

관찰에서는 "비 올 것 같다(Z) → 우산을 든다(X)"라는 경로가 살아 있어서 우산과 비가 함께 움직입니다. 하지만 **동전 던지기(무작위 배정)로** 우산 소지 여부를 정하면 Z→X 경로가 끊기므로, 우산 있는 그룹과 없는 그룹을 비교해 우산의 진짜 효과(비를 부르지 않음)가 드러납니다.

**구체적 예시: 약 복용과 회복**

```
증상 중증도 (Z, 교란변수)
    ↙          ↘
약 복용 (X)    회복 (Y)
    ↘          ↗
     (진짜 효과)
```

> **배경:** 100명의 환자가 모두 같은 병에 걸렸습니다. 하지만 **증상 중증도(Z)**는 경증~중증까지 다양합니다.

| | 관찰 P(회복 \| 약 복용) | 개입 P(회복 \| do(약 복용)) |
|---|---|---|
| **약 복용 그룹** | 중증 환자 비율 **높음** (심하게 아프니까 의사가 약 처방) | 경증·중증 **골고루** 섞임 (무작위 배정) |
| **비복용 그룹** | 경증 환자 비율 **높음** (가볍게 아프니까 약 없이 경과 관찰) | 경증·중증 **골고루** 섞임 (위약/플라시보 배정) |
| **비교 결과** | 약 효과 **과소추정** (약 복용 그룹에 중증이 몰려 회복률 낮아 보임) | 약의 **진짜 효과** |

**왜 이런 차이가 생기나요?**

관찰 데이터에서는 증상이 심한 환자일수록 의사가 약을 처방할 확률이 높습니다. 그래서 약 복용 그룹에 중증 환자가 몰립니다. 이 그룹의 회복률이 낮은 것은 약이 안 듣는 게 아니라, **원래 회복이 어려운 환자가 많기 때문**입니다.

개입(do)에서는 **모든 환자를** 대상으로 약 복용 여부를 **동전 던지기(무작위)로 배정**합니다. 절반은 진짜 약, 절반은 위약(플라시보)을 받습니다. 두 집단의 중증도 분포가 같아지므로, 회복률 차이가 곧 약의 **진짜 효과**입니다.

> 이것이 바로 **무작위 실험(RCT)**의 원리이고, **do-연산자**가 수학으로 표현하는 것입니다.  
> 실험이 불가능할 때는 **백도어 조정**(위의 공식)으로 관찰 데이터에서 do() 효과를 계산합니다.

---

## 5.4 반사실적 사고 (Counterfactual) — "만약 다르게 했다면?"

### 핵심 질문

> **"만약 그때 다르게 했다면 결과가 달라졌을까?"**

### 쉬운 예시: 우산을 안 가져갔더라면?

오늘 아침 우산을 가져갔더니 비를 안 맞았습니다.

- **실제 일어난 일**: 우산 O → 비 안 맞음
- **반사실 (일어나지 않은 가상)**: 우산 X → 비 맞았을까?

두 상황을 비교하면 **우산의 진짜 효과**를 알 수 있습니다. 하지만 문제가 있습니다:

> 같은 사람이 같은 날 우산을 가져가면서 동시에 안 가져갈 수는 없다!

이것이 인과 추론의 **근본적 문제**입니다. 한 사람에게 두 가지 결과를 동시에 볼 수 없기 때문이죠.

### 해결법: 비슷한 사람끼리 비교하기

한 사람의 두 결과를 볼 수 없으니, **비슷한 두 그룹**을 비교합니다.

| 방법 | 원리 |
|------|------|
| **무작위 실험 (RCT)** | 동전 던져서 처치/통제 나눔 → 두 그룹이 비슷해짐 |
| **교란변수 통제** | 같은 조건(나이, 성별 등)끼리 비교 → 숨은 원인 제거 |

### 평균 처치 효과 (ATE)

그룹 간 비교로 구하는 **평균적인 효과**:

**ATE** = (처치 그룹 평균 결과) - (통제 그룹 평균 결과)

- **무작위 실험**: 두 그룹이 동질 → 단순 비교 = ATE
- **관찰 데이터**: 그룹이 다를 수 있음 → 교란변수 통제 필요 (Part 4, 5에서 실습!)

---

## 5.5 준실험 설계: 이중차분법 (DID) — 실험 못 할 때 실험처럼 비교하기

> 무작위 실험이 불가능할 때, 관찰 데이터에서 **교묘하게** 진짜 효과를 추정하는 방법입니다.

### 한 줄 요약

> **"처치 그룹의 변화에서 통제 그룹의 변화를 빼면 순수한 정책 효과만 남는다."**

---

### 이중차분법이란?

**이중차분법 (Difference-in-Differences, DID)**은 정책이나 이벤트의 효과를 추정할 때, **시간에 따른 변화(1차 차이)**를 두 그룹 사이에서 **다시 비교(2차 차이)**하는 방법입니다.

```
        정책 전    정책 후      변화(1차 차이)
──────────────────────────────────────────────
처치 그룹   A₀   →   A₁         ΔA = A₁ - A₀  (자연추세 + 정책효과)
통제 그룹   B₀   →   B₁         ΔB = B₁ - B₀  (자연추세만)
──────────────────────────────────────────────
DID(2차 차이) = ΔA - ΔB = 순수 정책 효과
```

### 왜 필요한가?

단순히 "정책 전후 비교"만 하면, **정책과 상관없이 시간이 지나면서 자연히 변하는 추세**까지 정책 효과로 착각할 수 있습니다. 통제 그룹이 바로 그 자연 추세를 잡아주는 역할을 합니다.

### 전형적 예시: 최저임금 인상 → 고용 효과

> 1994년, 미국 뉴저지주가 최저임금을 인상했습니다. 인접한 펜실베이니아주는 인상하지 않았습니다.

| | 정책 전 고용 | 정책 후 고용 | 변화 |
|---|---|---|---|
| **뉴저지 (처치)** | 20.4명 | 21.0명 | +0.6 |
| **펜실베이니아 (통제)** | 23.3명 | 21.2명 | −2.1 |
| **DID** | | | 0.6 − (−2.1) = **+2.7** |

→ 최저임금 인상이 오히려 고용을 **2.7명** 늘렸다! (Card & Krueger, 1994)

### 핵심 가정: 평행 추세 (Parallel Trends)

> **"정책이 없었다면 두 그룹이 같은 추세를 보였을 것"**

```
결과
 ↑        처치 그룹 (관찰)  ──●
 │                         ╱
 │           정책 효과 ↕  ╱
 │                       ╱
 │     반사실 (점선) ──-●╱- - -
 │                     ╱
 │    통제 그룹 ──────●
 │
 └──────────────────────→ 시간
          ▲ 정책 시행
```

이 가정이 깨지면 DID 추정값은 편향됩니다. 따라서 **정책 전 기간**에 두 그룹의 추세가 실제로 비슷했는지 확인하는 것이 중요합니다.

---

In [5]:
# 이중차분법 (DID) 시뮬레이션
np.random.seed(42)

time = np.array([0, 1, 2, 3, 4, 5, 6, 7])
treatment_time = 4  # 정책 시행 시점

# 처치 그룹 (정책 대상)
treated_before = 20 + 2 * time[:treatment_time] + np.random.normal(0, 0.5, treatment_time)
treated_after = treated_before[-1] + 5 + 2 * np.arange(len(time[treatment_time:])) + np.random.normal(0, 0.5, len(time[treatment_time:]))
treated = np.concatenate([treated_before, treated_after])

# 통제 그룹 (정책 미대상)
control = 18 + 2 * time + np.random.normal(0, 0.5, len(time))

# 반사실 (정책 없었을 경우의 처치 그룹)
counterfactual = 20 + 2 * time + np.random.normal(0, 0.3, len(time))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=time, y=treated, mode='lines+markers',
    name='Treatment Group (Observed)',
    line=dict(color='black', width=3)
))

fig.add_trace(go.Scatter(
    x=time, y=control, mode='lines+markers',
    name='Control Group',
    line=dict(color='black', width=3)
))

fig.add_trace(go.Scatter(
    x=time[treatment_time:], y=counterfactual[treatment_time:],
    mode='lines', name='Counterfactual (No Policy)',
    line=dict(color='black', width=2, dash='dash')
))

# 정책 시행 시점
fig.add_vline(x=treatment_time - 0.5, line_dash='dash', line_color='gray',
              annotation_text='Policy Implementation')

# 진짜 효과 표시
fig.add_annotation(
    x=6, y=(treated[6] + counterfactual[6]) / 2,
    text=f'Causal Effect\n(DID = {treated[6] - counterfactual[6]:.1f})',
    showarrow=True, arrowhead=2, ax=50, ay=0,
    font=dict(color='red', size=12)
)

fig.update_layout(
    title='Difference-in-Differences (DID): Policy Effect Estimation',
    xaxis_title='Time Period',
    yaxis_title='Outcome',
    height=450
)

fig.show()

did_effect = (treated[-1] - treated[treatment_time-1]) - (control[-1] - control[treatment_time-1])
print(f"💡 DID 계산값 = (처치 후-전) - (통제 후-전) = {did_effect:.1f}")
print("  = 처치그룹 변화에서 자연적 추세(통제그룹)를 빼면 순수 정책 효과")
print("  핵심 가정: 정책 없었으면 두 그룹이 같은 추세를 보였을 것 (평행 추세)")


💡 DID 계산값 = (처치 후-전) - (통제 후-전) = 3.4
  = 처치그룹 변화에서 자연적 추세(통제그룹)를 빼면 순수 정책 효과
  핵심 가정: 정책 없었으면 두 그룹이 같은 추세를 보였을 것 (평행 추세)


---

## Part 4. 실습: 마케팅 캠페인의 진짜 효과 찾기

---

### 시나리오

> 마케팅팀장이 요청합니다:
>
> *"지난 분기에 이메일 캠페인을 진행했는데, 효과가 있었는지 분석해주세요.
> 캠페인을 받은 고객과 안 받은 고객의 구매율을 비교해주면 됩니다."*

### 데이터: `data/campaign.csv` (1,000명 고객 × 2기간 = 2,000행)

| 열 이름 | 설명 |
|---------|------|
| `customer_id` | 고객 고유번호 |
| `period` | 기간 (0 = 캠페인 전, 1 = 캠페인 후) |
| `campaign` | 캠페인 대상 여부 (0 = 비대상, 1 = 대상) |
| `customer_value` | 고객 구매성향 점수 (0~1) — **교란변수** |
| `recency` | 마지막 구매 후 경과 일수 |
| `frequency` | 과거 구매 횟수 |
| `purchase` | 구매 여부 (0/1) |

> **주의:** `customer_value`가 높은 고객일수록 캠페인 대상이 될 확률이 높고, 원래 구매 확률도 높습니다(교란변수). 단순 전후 비교나 단순 그룹 비교만으로는 부족합니다!

---

### AI에게 이렇게 질문하세요

아래 내용을 복사해서 AI(Claude, ChatGPT 등)에게 붙여넣으세요:

> 이메일 캠페인의 진짜 효과를 이중차이분석(DID)으로 추정하고 싶어.
> 데이터는 고객 1,000명의 캠페인 전후(period=0/1) 패널 데이터야.
> customer_value가 교란변수로, 캠페인 대상 배정과 구매 모두에 영향을 줘.
> 어떤 순서로 분석해야 하는지 알려줘.
> 데이터에는 customer_id, period, campaign, customer_value, recency, frequency, purchase 열이 있어.

AI가 분석 순서를 알려주면, **순서대로 하나씩** 진행하세요:

1. 각 단계마다 **"코드를 작성해줘"**라고 요청
2. 코드를 아래 셀에 붙여넣고 실행
3. 결과가 나오면 **"이 결과를 해석해줘"**라고 요청
4. 다음 단계로 넘어가기

In [4]:
# 데이터 로드
df = pd.read_csv('data/campaign.csv')
print(f"데이터 크기: {df.shape}")
df.head()

데이터 크기: (2000, 7)


,customer_id,period,campaign,customer_value,recency,frequency,purchase
0,0,0,0,0.3871,2.0,5,1
1,1,0,1,0.9056,189.2,4,1
2,2,0,1,0.7088,13.9,2,0
3,3,0,0,0.5888,54.4,5,0
4,4,0,1,0.1904,91.3,8,0


In [8]:
# ✏️ AI가 제안한 분석 단계를 순서대로 실행하세요 (1단계)


In [9]:
# ✏️ 2단계


In [10]:
# ✏️ 3단계


### 참고

- 이 데이터의 실제 캠페인 효과(True ATE)는 **0.15** (구매 확률 15%p 증가)
- 단순 DID 추정값은 약 **0.19** — 교란변수(`customer_value`)를 통제하면 0.15에 더 가까워짐
- **도전과제:** 회귀 DID(period, campaign, period×campaign + 공변량)로 추정값이 개선되는지 확인해보세요

---

---

## Part 5. 실습: 디지털 전환 투자의 효과 분해

---

### 시나리오

> 전략팀장이 요청합니다:
>
> *"디지털 전환 투자가 실제로 ROE를 개선하는지 분석해주세요.
> 특히 프로세스 효율을 높여서 성과가 나는 건지, 다른 경로인지도 알고 싶습니다."*

### 인과 구조 (DAG)

```
기업규모 (Z₁) ──┐          ┌──→ 프로세스 효율 (M) ──→ ROE 개선 (Y)
                ├──→ 디지털 투자 (X) ──┤
산업특성 (Z₂) ──┘          └──────── 직접 효과 ────────→ ROE 개선 (Y)
```

### 데이터: `data/digital_transform.csv` (500개 기업)

| 열 이름 | 설명 |
|---------|------|
| `firm_size` | 기업규모 (매출액) |
| `industry_tech` | IT산업 여부 (0/1) |
| `digital_invest` | 디지털 투자 수준 (0~15) |
| `process_eff` | 프로세스 효율성 점수 |
| `roe_improvement` | ROE 개선 (%p) |

---

### AI에게 이렇게 질문하세요

> 디지털 투자가 ROE 개선에 미치는 효과를 분석하고 싶어.
> 교란변수(기업규모, 산업특성)와 매개변수(프로세스 효율)가 있어.
> 총 효과, 직접 효과, 간접 효과를 분해하려면 어떤 순서로 분석해야 해?
> 데이터에는 firm_size, industry_tech, digital_invest, process_eff, roe_improvement 열이 있어.

위와 같은 방식으로 진행하세요: AI 제안 → 코드 요청 → 실행 → 해석 요청

In [11]:
# 데이터 로드
df_dt = pd.read_csv('data/digital_transform.csv')
print(f"데이터 크기: {df_dt.shape}")
df_dt.head()

데이터 크기: (500, 5)


,firm_size,industry_tech,digital_invest,process_eff,roe_improvement
0,243.89,1,4.91,1.11,2.63
1,129.25,0,2.42,0.24,1.94
2,283.64,0,4.70,3.55,2.29
3,680.64,1,4.05,1.37,2.83
4,117.43,1,5.85,1.88,3.86


In [12]:
# ✏️ AI가 제안한 분석 단계를 순서대로 실행하세요 (1단계)

In [13]:
# ✏️ 2단계


In [14]:
# ✏️ 3단계


### 참고

- 이 데이터의 실제 효과: 총 효과 = **0.25**, 직접 효과 = **0.08**, 간접 효과 = **0.17**
- 간접 효과(프로세스 경유)가 총 효과의 약 68%
- 시사점: 디지털 투자와 프로세스 재설계가 함께 이루어져야 효과 극대화

---

---

## 🎓 강의 마무리 및 핵심 요약

### 오늘 배운 내용

1. **상관관계 ≠ 인과관계**: 교란변수에 의한 가짜 관계, Simpson's Paradox에 주의
2. **DAG로 인과구조 시각화**: 교란변수(통제 필수), 매개변수(상황에 따라), 충돌변수(통제 금지 → Collider Bias)
3. **See vs Do**: 관찰과 개입은 다르다 → do-연산자로 진짜 효과 정의
4. **준실험 설계**: DID, RDD로 관찰 데이터에서도 인과 추정 가능
5. **교란변수 통제의 위력**: 단순 비교 vs 통제 비교의 극적 차이를 시뮬레이션으로 확인

### 핵심 개념 요약

| 개념 | 핵심 원칙 | 실무 적용 |
|------|----------|----------|
| 상관 ≠ 인과 | 함께 움직임 ≠ 유발 | 정책 설계 시 인과관계 확인 필수 |
| 교란변수 | 처치와 결과 모두에 영향 | DAG로 식별 → 통제 |
| 충돌변수 | 처치와 결과 모두의 결과 | 통제하면 Collider Bias 발생 → 통제 금지 |
| do-연산자 | 개입의 효과를 정의 | RCT 또는 백도어 조정 |
| 효과 분해 | 총 = 직접 + 간접 | 매개경로의 중요성 파악 |

### 핵심 메시지

> 기획은 **"무엇을 하면 어떤 결과가 나올까?"**에 답하는 것이다.  
> 이 질문에 제대로 답하려면 상관관계가 아닌 **인과관계**를 이해해야 한다.  
> 원인-결과 찾기 없는 기획은 잘못된 방향으로 자원을 낭비할 위험이 있다.

### 3-4장과의 연결

- **3장(로직 트리)**: Why Tree에서 도출한 원인 가설 → 5장에서 원인-결과 찾기으로 **검증**
- **4장(이슈 정의)**: "디지털 역량 부족이 매출 정체의 원인인가?" → 상관? 인과? 검증 필요
- **DAG**: 로직 트리의 가정(A→B→C)을 **인과 구조로 명시화**

### 다음 장 예고

**제6장: 시스템 다이내믹스**
- 5장의 **선형적** 인과관계 → 6장의 **피드백 루프**를 포함한 동적 시스템으로 확장
- 캠페인 → 구매 → 재구매 → 캠페인 노출 증가 (순환적 관계)
- DAG는 비순환이지만, 현실의 많은 시스템은 순환적

### 📚 추가 학습 자료

- **Judea Pearl & Dana Mackenzie** (2018). *The Book of Why*. Basic Books.
- **Miguel Hernán & James Robins** (2020). *Causal Inference: What If*. CRC Press.
- **Scott Cunningham** (2021). *Causal Inference: The Mixtape*. Yale University Press.
- Python: `dowhy` (원인-결과 찾기), `econml` (인과 머신러닝), `causal-learn` (인과 발견)

---

**제5장 끝**